# Initial Cleaning of the Merged Master Dataset

This notebook performs the first stage of cleaning and inspection for the merged thesis dataset.

It is designed to:
- load the merged dataset
- inspect column names, data types, and general data quality
- review columns in a structured way
- apply only basic and safe cleaning steps
- avoid model-specific preprocessing at this stage

The goal is to make the dataset easier to understand before any later feature engineering, imputation strategy, encoding, or modeling decisions.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

## Load Dataset

The merged dataset combines price observations with Traficom-based features. A copy of the raw data is kept so that all cleaning steps remain explicit and easy to review.

In [2]:
data_path = Path("/Users/riteshbhandari/Documents/Dokumentit – Ritesh - MacBook Pro/GitHub/DPPM/datasets/merged/price_traficom_merged.csv")

if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found: {data_path}")

df_raw = pd.read_csv(data_path, skipinitialspace=True)
df = df_raw.copy()

print(f"Dataset path: {data_path}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

display(df.head())

Dataset path: /Users/riteshbhandari/Documents/Dokumentit – Ritesh - MacBook Pro/GitHub/DPPM/datasets/merged/price_traficom_merged.csv
Rows: 11,374
Columns: 70


,product_id,part_name,price,quality_grade,oem_number,mileage,brand,model,category,subcategory,scrape_date,year_start,year_end,year_span,year_mid,brand_merge_key,model_merge_key,repair_status,brand_is_known_model_family,model_looks_like_part_taxonomy,model_family_clean,model_total_registered,model_median_vehicle_age,model_mean_vehicle_age,model_median_mileage,model_mean_mileage,model_median_engine_cc,model_median_power_kw,model_median_mass_kg,model_share_of_market,model_share_within_brand,model_share_over_10y,model_share_over_200k_km,model_automatic_share,model_petrol_share,model_diesel_share,model_ev_share,model_hybrid_share,model_turbo_share,model_firstreg_total_2014_2026,model_firstreg_year_span,model_firstreg_peak_year,model_firstreg_peak_count,model_firstreg_recent_share,model_firstreg_old_share,model_firstreg_weighted_year,brand_total_registered,brand_median_vehicle_age,brand_mean_vehicle_age,brand_median_mileage,brand_mean_mileage,brand_median_engine_cc,brand_median_power_kw,brand_median_mass_kg,brand_share_of_market,brand_share_over_10y,brand_share_over_200k_km,brand_automatic_share,brand_petrol_share,brand_diesel_share,brand_ev_share,brand_hybrid_share,brand_turbo_share,brand_firstreg_total_2014_2026,brand_firstreg_year_span,brand_firstreg_peak_year,brand_firstreg_peak_count,brand_firstreg_recent_share,brand_firstreg_old_share,brand_firstreg_weighted_year
0,"63,136,980.000","Contact roll Airbag - , e-",177.600,A2,FI02042722A,"224,000.000",vw,golf,airbag,contact roll airbag,2026-02-03,2013,2020,7,"2,016.500",volkswagen,golf,original_valid,True,False,golf,86762,13.000,13.749,"186,801.000","195,467.968","1,395.000",81.000,"1,330.000",0.035,0.317,0.652,0.433,0.383,0.786,0.170,0.013,0.068,0.726,"31,752.000",11.000,"2,014.000","5,265.000",0.167,0.671,"2,017.576",279311,11.000,12.154,"177,745.000","192,564.366","1,498.000",85.000,"1,400.000",0.109,0.538,0.387,0.478,0.657,0.246,0.075,0.073,0.686,"100,381.000",9.000,"2,018.000","13,274.000",0.393,0.391,"2,019.842"
1,"64,390,201.000","Contact roll Airbag - , e-",177.600,A2,FI05351686A,"272,000.000",vw,golf,airbag,contact roll airbag,2026-02-03,2013,2020,7,"2,016.500",volkswagen,golf,original_valid,True,False,golf,86762,13.000,13.749,"186,801.000","195,467.968","1,395.000",81.000,"1,330.000",0.035,0.317,0.652,0.433,0.383,0.786,0.170,0.013,0.068,0.726,"31,752.000",11.000,"2,014.000","5,265.000",0.167,0.671,"2,017.576",279311,11.000,12.154,"177,745.000","192,564.366","1,498.000",85.000,"1,400.000",0.109,0.538,0.387,0.478,0.657,0.246,0.075,0.073,0.686,"100,381.000",9.000,"2,018.000","13,274.000",0.393,0.391,"2,019.842"
2,"58,952,159.000","Contact roll Airbag - , e-",177.600,A2,FI27837687A,"134,000.000",vw,golf,airbag,contact roll airbag,2026-02-03,2013,2020,7,"2,016.500",volkswagen,golf,original_valid,True,False,golf,86762,13.000,13.749,"186,801.000","195,467.968","1,395.000",81.000,"1,330.000",0.035,0.317,0.652,0.433,0.383,0.786,0.170,0.013,0.068,0.726,"31,752.000",11.000,"2,014.000","5,265.000",0.167,0.671,"2,017.576",279311,11.000,12.154,"177,745.000","192,564.366","1,498.000",85.000,"1,400.000",0.109,0.538,0.387,0.478,0.657,0.246,0.075,0.073,0.686,"100,381.000",9.000,"2,018.000","13,274.000",0.393,0.391,"2,019.842"
3,"63,225,719.000","Contact roll Airbag - , e-",177.600,A2,FI27837687A,"253,000.000",vw,golf,airbag,contact roll airbag,2026-02-03,2013,2020,7,"2,016.500",volkswagen,golf,original_valid,True,False,golf,86762,13.000,13.749,"186,801.000","195,467.968","1,395.000",81.000,"1,330.000",0.035,0.317,0.652,0.433,0.383,0.786,0.170,0.013,0.068,0.726,"31,752.000",11.000,"2,014.000","5,265.000",0.167,0.671,"2,017.576",279311,11.000,12.154,"177,745.000","192,564.366","1,498.000",85.000,"1,400.000",0.109,0.538,0.387,0.478,0.657,0.246,0.075,0.073,0.686,"100,381.000",9.000,"2,018.000","13,274.000",0.393,0.391,"2,019.842"
4,"64,336,640.000","Contact roll Airbag - , e-",177.600,A2,FI09389104A,"152,000.000",vw,golf,airbag,contact roll airbag,2026-02-03,2013,2020,7,"2,016.500",volkswagen,

## Initial Overview

This first overview checks dataset size, column names, and the starting data types before any cleaning changes are applied.

In [3]:
print("Original column names:")
print(df.columns.tolist())
print()

df.info()

Original column names:
['product_id', 'part_name', 'price', 'quality_grade', 'oem_number', 'mileage', 'brand', 'model', 'category', 'subcategory', 'scrape_date', 'year_start', 'year_end', 'year_span', 'year_mid', 'brand_merge_key', 'model_merge_key', 'repair_status', 'brand_is_known_model_family', 'model_looks_like_part_taxonomy', 'model_family_clean', 'model_total_registered', 'model_median_vehicle_age', 'model_mean_vehicle_age', 'model_median_mileage', 'model_mean_mileage', 'model_median_engine_cc', 'model_median_power_kw', 'model_median_mass_kg', 'model_share_of_market', 'model_share_within_brand', 'model_share_over_10y', 'model_share_over_200k_km', 'model_automatic_share', 'model_petrol_share', 'model_diesel_share', 'model_ev_share', 'model_hybrid_share', 'model_turbo_share', 'model_firstreg_total_2014_2026', 'model_firstreg_year_span', 'model_firstreg_peak_year', 'model_firstreg_peak_count', 'model_firstreg_recent_share', 'model_firstreg_old_share', 'model_firstreg_weighted_year',

## Duplicate Observation Warning

Some observations that may look like duplicates can represent the **same listing or part observed repeatedly over time**. In this project, these repeated observations should be preserved unless there is a very clear accidental exact-duplicate issue.

For that reason, this notebook preserves repeated observations across **different scrape dates**. It only removes clear same-day duplicate listing rows where the same `product_id` appears more than once on the same `scrape_date`.

## Basic Cleaning

The cleaning below is intentionally conservative. It standardizes names and obvious formatting issues without making modeling decisions or deleting information.

In [4]:
original_columns = df.columns.tolist()

# Standardize column names for easier handling.
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
)

object_columns = df.select_dtypes(include=["object"]).columns.tolist()
for column in object_columns:
    df[column] = df[column].astype("string").str.strip()

# Replace empty strings with missing values after whitespace stripping.
df[object_columns] = df[object_columns].replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})

numeric_name_fragments = [
    "price",
    "mileage",
    "year",
    "age",
    "registered",
    "engine_cc",
    "power_kw",
    "mass_kg",
    "share",
    "span",
    "mid",
]

candidate_numeric_columns = [
    column
    for column in df.columns
    if any(fragment in column for fragment in numeric_name_fragments)
    and column not in {"product_id"}
]

for column in candidate_numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

date_like_columns = [column for column in df.columns if "date" in column]
for column in date_like_columns:
    df[column] = pd.to_datetime(df[column], errors="coerce")

if "product_id" in df.columns:
    product_id_numeric = pd.to_numeric(df["product_id"], errors="coerce")
    df["product_id"] = product_id_numeric.astype("Int64").astype("string")

same_day_duplicate_group_count = 0
same_day_duplicate_row_count = 0
same_day_duplicate_rows_removed = 0
if {"product_id", "scrape_date"}.issubset(df.columns):
    duplicate_mask = df.duplicated(subset=["product_id", "scrape_date"], keep=False)
    same_day_duplicate_row_count = int(duplicate_mask.sum())
    same_day_duplicate_group_count = int(
        df.loc[duplicate_mask, ["product_id", "scrape_date"]].drop_duplicates().shape[0]
    )
    rows_before_same_day_dedup = len(df)

    if same_day_duplicate_row_count > 0:
        df["_row_order"] = np.arange(len(df))
        df["_non_missing_count"] = df.notna().sum(axis=1)
        df = (
            df.sort_values(["_non_missing_count", "_row_order"], ascending=[False, True])
            .drop_duplicates(subset=["product_id", "scrape_date"], keep="first")
            .sort_values("_row_order")
            .drop(columns=["_row_order", "_non_missing_count"])
            .reset_index(drop=True)
        )
        same_day_duplicate_rows_removed = rows_before_same_day_dedup - len(df)

print(f"Same-day duplicate groups removed: {same_day_duplicate_group_count}")
print(f"Same-day duplicate rows removed: {same_day_duplicate_rows_removed}")

print("Column names before cleaning:")
print(original_columns)
print()
print("Column names after cleaning:")
print(df.columns.tolist())
print()
print(f"Object/string columns cleaned: {len(object_columns)}")
print(f"Numeric columns coerced where obvious: {len(candidate_numeric_columns)}")
print(f"Date-like columns parsed where obvious: {len(date_like_columns)}")

Same-day duplicate groups removed: 53
Same-day duplicate rows removed: 53
Column names before cleaning:
['product_id', 'part_name', 'price', 'quality_grade', 'oem_number', 'mileage', 'brand', 'model', 'category', 'subcategory', 'scrape_date', 'year_start', 'year_end', 'year_span', 'year_mid', 'brand_merge_key', 'model_merge_key', 'repair_status', 'brand_is_known_model_family', 'model_looks_like_part_taxonomy', 'model_family_clean', 'model_total_registered', 'model_median_vehicle_age', 'model_mean_vehicle_age', 'model_median_mileage', 'model_mean_mileage', 'model_median_engine_cc', 'model_median_power_kw', 'model_median_mass_kg', 'model_share_of_market', 'model_share_within_brand', 'model_share_over_10y', 'model_share_over_200k_km', 'model_automatic_share', 'model_petrol_share', 'model_diesel_share', 'model_ev_share', 'model_hybrid_share', 'model_turbo_share', 'model_firstreg_total_2014_2026', 'model_firstreg_year_span', 'model_firstreg_peak_year', 'model_firstreg_peak_count', 'model_fi

## Column Summary

A compact summary table helps identify the role and quality of each column. The conceptual column type is partly heuristic, so it should be treated as a review aid rather than a final classification.

In [5]:
def conceptual_column_type(series: pd.Series) -> str:
    column_name = series.name.lower()

    if column_name == "price":
        return "target"
    if column_name.endswith("_id") or column_name == "product_id":
        return "id"
    if pd.api.types.is_datetime64_any_dtype(series):
        return "date"
    if pd.api.types.is_numeric_dtype(series):
        return "numeric"
    if any(token in column_name for token in ["date", "time"]):
        return "date"
    if any(token in column_name for token in ["name", "description", "comment", "text", "number"]):
        return "text"
    if series.nunique(dropna=True) <= 30:
        return "categorical"
    return "text"


def sample_values(series: pd.Series, max_values: int = 5) -> str:
    values = series.dropna().astype(str).unique().tolist()[:max_values]
    return ", ".join(values)


column_summary = pd.DataFrame(
    {
        "column_name": df.columns,
        "dtype": [str(df[column].dtype) for column in df.columns],
        "conceptual_type": [conceptual_column_type(df[column]) for column in df.columns],
        "missing_count": [int(df[column].isna().sum()) for column in df.columns],
        "missing_percentage": [df[column].isna().mean() * 100 for column in df.columns],
        "unique_values": [int(df[column].nunique(dropna=True)) for column in df.columns],
        "sample_values": [sample_values(df[column]) for column in df.columns],
    }
).sort_values(["conceptual_type", "column_name"]).reset_index(drop=True)


display(column_summary)

,column_name,dtype,conceptual_type,missing_count,missing_percentage,unique_values,sample_values
0,brand,string,categorical,0,0.000,3,"vw, toyota, skoda"
1,brand_merge_key,string,categorical,0,0.000,3,"volkswagen, toyota, skoda"
2,category,string,categorical,0,0.000,7,"airbag, brakes, electric / transmitter / databox / sensor, engine, fuel"
3,model,string,categorical,0,0.000,3,"golf, corolla, octavia"
4,model_family_clean,string,categorical,0,0.000,3,"golf, corolla, octavia"
...,...,...,...,...,...,...,...
65,year_start,int64,numeric,0,0.000,16,"2013, 1992, 2004, 2020, 2009"
66,price,float64,target,0,0.000,1135,"177.6, 23.7, 17.8, 14.2, 473.6"
67,oem_number,string,text,0,0.000,54,"FI02042722A, FI05351686A, FI27837687A, FI09389104A, FI03986645A"
68,part_name,string,text,0,0.000,486,"Contact roll Airbag - , e-, Other airbags - , e-, Passenger airbag - , e-, Passenger airbag - , e-(Interior front), ..."


## Column-by-Column Inspection

The next cells inspect the dataset by column type. This is useful for spotting unusual ranges, unexpected values, and columns that may need more careful treatment later.

In [6]:
numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()

numeric_summary = pd.DataFrame(
    {
        "column_name": numeric_columns,
        "missing_count": [int(df[column].isna().sum()) for column in numeric_columns],
        "missing_percentage": [df[column].isna().mean() * 100 for column in numeric_columns],
        "min": [df[column].min() for column in numeric_columns],
        "max": [df[column].max() for column in numeric_columns],
        "mean": [df[column].mean() for column in numeric_columns],
        "median": [df[column].median() for column in numeric_columns],
    }
).sort_values("column_name")

print(f"Numeric columns: {len(numeric_columns)}")
display(numeric_summary)

Numeric columns: 55


,column_name,missing_count,missing_percentage,min,max,mean,median
42,brand_automatic_share,0,0.000,0.024,0.592,0.358,0.478
44,brand_diesel_share,0,0.000,0.069,0.269,0.193,0.246
45,brand_ev_share,0,0.000,0.005,0.075,0.048,0.067
53,brand_firstreg_old_share,0,0.000,0.308,0.391,0.353,0.363
51,brand_firstreg_peak_count,0,0.000,"11,697.000","15,184.000","13,424.339","13,274.000"
50,brand_firstreg_peak_year,0,0.000,"2,018.000","2,019.000","2,018.344","2,018.000"
52,brand_firstreg_recent_share,0,0.000,0.393,0.485,0.431,0.411
48,brand_firstreg_total_2014_2026,0,0.000,"93,798.000","137,879.000","111,164.044","100,381.000"
54,brand_firstreg_weighted_year,0,0.000,"2,019.842","2,020.379","2,020.081","2,020.009"
49,brand_firstreg_year_span,0,0.000,9.000,9.000,9.000,9.000


In [7]:
categorical_columns = [
    column
    for column in df.columns
    if df[column].dtype in ["object", "string", "category", "boolean"]
]

print(f"Categorical/text-like columns selected for quick review: {len(categorical_columns)}")

for column in categorical_columns:
    print(f"\n--- {column} ---")
    print(f"Missing values: {df[column].isna().sum():,} ({df[column].isna().mean() * 100:.2f}%)")
    display(df[column].value_counts(dropna=False).head(10).rename("count").to_frame())


Categorical/text-like columns selected for quick review: 12

--- product_id ---
Missing values: 0 (0.00%)


,count
product_id,
56848565,6
60557555,6
65651367,6
65037599,6
53409488,6
53379281,6
62662904,6
53397794,6
65651373,6



--- part_name ---
Missing values: 0 (0.00%)


,count
part_name,
Suspension -(Rear),135
Trailing link rear -(Left),128
Shock absorbers rear -(Rear),128
Drive shaft -(Left front),128
Curtain airbags -(Right),117
Drive shaft -(Right front),112
Trailing link rear -(Right),110
Wheel bearing spindle shaft -(Left rear),106
Hub rear -(Rear),103



--- quality_grade ---
Missing values: 0 (0.00%)


...

,count
model_merge_key,
corolla,3894
golf,3790
octavia,3637



--- repair_status ---
Missing values: 0 (0.00%)


,count
repair_status,
original_valid,11321



--- model_family_clean ---
Missing values: 0 (0.00%)


,count
model_family_clean,
corolla,3894
golf,3790
octavia,3637


In [8]:
date_columns = [column for column in df.columns if pd.api.types.is_datetime64_any_dtype(df[column])]

if date_columns:
    date_summary = pd.DataFrame(
        {
            "column_name": date_columns,
            "missing_count": [int(df[column].isna().sum()) for column in date_columns],
            "missing_percentage": [df[column].isna().mean() * 100 for column in date_columns],
            "min_date": [df[column].min() for column in date_columns],
            "max_date": [df[column].max() for column in date_columns],
        }
    ).sort_values("column_name")
    display(date_summary)
else:
    print("No date columns were parsed successfully in the basic cleaning step.")

,column_name,missing_count,missing_percentage,min_date,max_date
0,scrape_date,0,0.000,2026-02-03,2026-02-18


## Duplicate Checks

These checks are purely diagnostic. They help assess whether there are exact row duplicates or repeated identifiers, but no rows are removed automatically.

In [9]:
full_row_duplicates = int(df.duplicated().sum())
print(f"Fully duplicated rows: {full_row_duplicates:,}")

id_columns = [column for column in df.columns if column.endswith("_id") or column == "product_id"]

if id_columns:
    duplicate_id_summary = pd.DataFrame(
        {
            "column_name": id_columns,
            "duplicate_values": [int(df[column].duplicated(keep=False).sum()) for column in id_columns],
            "unique_values": [int(df[column].nunique(dropna=True)) for column in id_columns],
            "missing_count": [int(df[column].isna().sum()) for column in id_columns],
        }
    ).sort_values("column_name")
    display(duplicate_id_summary)
else:
    print("No ID-like columns were detected.")

Fully duplicated rows: 0


,column_name,duplicate_values,unique_values,missing_count
0,product_id,10487,2619,0


## Missing Values Review

Missing values are reviewed after the basic cleaning step so that empty strings and parsing failures are already reflected in the totals.

In [10]:
missing_values_review = (
    pd.DataFrame(
        {
            "column_name": df.columns,
            "missing_count": [int(df[column].isna().sum()) for column in df.columns],
            "missing_percentage": [df[column].isna().mean() * 100 for column in df.columns],
        }
    )
    .sort_values(["missing_percentage", "missing_count"], ascending=[False, False])
    .reset_index(drop=True)
)

print("Columns with the highest missingness:")
display(missing_values_review.head(20))

Columns with the highest missingness:


,column_name,missing_count,missing_percentage
0,mileage,1107,9.778
1,product_id,0,0.000
2,part_name,0,0.000
3,price,0,0.000
4,quality_grade,0,0.000
5,oem_number,0,0.000
6,brand,0,0.000
7,model,0,0.000
8,category,0,0.000
9,subcategory,0,0.000


## Consistency Checks

The following quick checks are simple diagnostics for obvious structural issues in the cleaned dataset. They are kept separate from the main summary sections so the notebook remains easier to follow.

In [11]:
# Logical year consistency
((df["year_end"] < df["year_start"]).sum(), (df["year_span"] <= 0).sum())

(np.int64(0), np.int64(0))

## Mileage Review

Mileage deserves a closer look because it is one of the most practically important variables in the dataset and may contain unusually small values that require cautious handling.

In [12]:
# Mileage tail inspection
df["mileage"].quantile([0, 0.01, 0.05, 0.5, 0.95, 0.99, 1.0])

0.000         1.000
0.010     2,000.000
0.050    16,000.000
0.500   120,000.000
0.950   324,000.000
0.990   433,000.000
1.000   919,294.000
Name: mileage, dtype: float64

In [13]:
# Correlation quick look for numeric columns
numeric_df = df.select_dtypes(include=["number"])
numeric_df.corr(numeric_only=True)["price"].sort_values(ascending=False)

price                             1.000
year_mid                          0.227
year_end                          0.226
year_start                        0.225
year_span                         0.099
model_diesel_share                0.021
model_median_mass_kg              0.021
model_automatic_share             0.021
brand_median_power_kw             0.021
model_turbo_share                 0.020
brand_automatic_share             0.020
model_share_within_brand          0.020
brand_turbo_share                 0.020
brand_diesel_share                0.020
model_median_power_kw             0.018
brand_ev_share                    0.018
model_firstreg_old_share          0.017
model_firstreg_total_2014_2026    0.016
model_firstreg_peak_count         0.016
brand_firstreg_old_share          0.014
brand_median_mass_kg              0.012
model_ev_share                    0.001
model_median_engine_cc           -0.001
model_mean_mileage               -0.010
brand_firstreg_weighted_year     -0.015


In [14]:
df.loc[df["mileage"] <= 1000, ["product_id", "part_name", "mileage", "brand", "model", "scrape_date"]]

,product_id,part_name,mileage,brand,model,scrape_date
122,65711229,"Adaptiv Farthållare Sensor - , e-","1,000.000",vw,golf,2026-02-03
154,54070103,"Other Control unit - , e-","1,000.000",vw,golf,2026-02-03
186,54410773,"Battery box / Fixing / Holder - , e-",1.000,vw,golf,2026-02-03
202,53919106,"Sensor ABS - , e-","1,000.000",vw,golf,2026-02-03
203,53919107,"Sensor ABS - , e-","1,000.000",vw,golf,2026-02-03
756,65711229,"Adaptiv Farthållare Sensor - , e-","1,000.000",vw,golf,2026-02-06
788,54070103,"Other Control unit - , e-","1,000.000",vw,golf,2026-02-06
820,54410773,"Battery box / Fixing / Holder - , e-",1.000,vw,golf,2026-02-06
836,53919106,"Sensor ABS - , e-","1,000.000",vw,golf,2026-02-06
837,53919107,"Sensor ABS - , e-","1,000.000",vw,golf,2026-02-06


In [15]:
low_mileage_count = (df["mileage"] <= 1000).sum()
print(low_mileage_count)

49


In [16]:
# Replace implausibly low mileage values (<= 1000) with missing values
df.loc[df["mileage"] <= 1000, "mileage"] = np.nan

# Check updated mileage summary
print(df["mileage"].describe())

count    10,165.000
mean    145,544.167
std     101,899.615
min       2,000.000
25%      68,000.000
50%     121,000.000
75%     204,000.000
max     919,294.000
Name: mileage, dtype: float64


## Mileage Imputation

Missing mileage is handled here before grouped splitting so that every downstream notebook uses the same cleaned value. A missingness flag is retained, and mileage is filled with a hierarchical median: first by brand and model, then by brand, and finally by the overall dataset median.

In [17]:
# Preserve whether mileage was originally missing after the low-value cleanup.
df["mileage_missing_flag"] = df["mileage"].isna().astype(int)

# Build stable reference medians from the cleaned dataset.
global_mileage_median = df["mileage"].median()
brand_model_mileage_median = (
    df.groupby(["brand", "model"], dropna=False)["mileage"]
    .median()
    .rename("brand_model_mileage_median")
)
brand_mileage_median = (
    df.groupby("brand", dropna=False)["mileage"]
    .median()
    .rename("brand_mileage_median")
)

df = df.merge(
    brand_model_mileage_median,
    on=["brand", "model"],
    how="left",
)
df = df.merge(
    brand_mileage_median,
    on="brand",
    how="left",
)


In [18]:
# Fill missing mileage from the most specific median available.
df["mileage"] = df["mileage"].fillna(df["brand_model_mileage_median"])
df["mileage"] = df["mileage"].fillna(df["brand_mileage_median"])
df["mileage"] = df["mileage"].fillna(global_mileage_median)

df = df.drop(columns=["brand_model_mileage_median", "brand_mileage_median"])

mileage_imputation_summary = pd.DataFrame(
    {
        "metric": [
            "rows originally missing mileage",
            "rows still missing mileage after imputation",
            "global mileage median fallback",
        ],
        "value": [
            int(df["mileage_missing_flag"].sum()),
            int(df["mileage"].isna().sum()),
            float(global_mileage_median),
        ],
    }
)

display(mileage_imputation_summary)
print(df[["mileage", "mileage_missing_flag"]].describe())


,metric,value
0,rows originally missing mileage,"1,156.000"
1,rows still missing mileage after imputation,0.000
2,global mileage median fallback,"121,000.000"


          mileage  mileage_missing_flag
count  11,321.000            11,321.000
mean  143,050.743                 0.102
std    96,840.644                 0.303
min     2,000.000                 0.000
25%    75,000.000                 0.000
50%   122,000.000                 0.000
75%   194,000.000                 0.000
max   919,294.000                 1.000


In [19]:
# Inspect very low mileage rows
print(df.loc[df["mileage"] <= 5000, ["product_id", "part_name", "mileage", "brand", "model", "scrape_date"]].sort_values("mileage"))

      product_id                               part_name   mileage   brand  \
494     54247541          Drive shaft - , e-(Left front) 2,000.000      vw   
587     54254408  Trailing link rear - , e-(Right upper) 2,000.000      vw   
1220    54254408  Trailing link rear - , e-(Right upper) 2,000.000      vw   
1127    54247541          Drive shaft - , e-(Left front) 2,000.000      vw   
1748    54247541          Drive shaft - , e-(Left front) 2,000.000      vw   
...          ...                                     ...       ...     ...   
9911    53489119                              Tank lid - 4,000.000   skoda   
8702    53489119                              Tank lid - 4,000.000   skoda   
9304    53489119                              Tank lid - 4,000.000   skoda   
7405    53848511                       Engine Gasoline - 4,000.000  toyota   
11126   53489119                              Tank lid - 4,000.000   skoda   

         model scrape_date  
494       golf  2026-02-03  
587  

In [20]:
# Low-end mileage frequency table up to 20,000
print(df.loc[df["mileage"] <= 20000, "mileage"].value_counts().sort_index())

mileage
2,000.000     79
2,334.000     18
3,000.000      6
4,000.000     31
6,000.000     12
7,000.000     59
8,000.000     32
10,000.000    40
11,000.000    66
12,000.000    60
15,000.000    37
16,000.000    72
17,000.000    56
18,000.000    62
19,000.000    30
20,000.000    88
Name: count, dtype: int64


In [21]:
# Summary counts by threshold
for threshold in [1000, 2000, 3000, 5000, 7000, 10000, 15000, 20000]:
    count = (df["mileage"] <= threshold).sum()
    print(f"<= {threshold}: {count}")

<= 1000: 0
<= 2000: 79
<= 3000: 103
<= 5000: 134
<= 7000: 205
<= 10000: 277
<= 15000: 440
<= 20000: 748


In [22]:
# Inspect rows in the suspicious low range
print(
    df.loc[
        df["mileage"].between(1001, 10000),
        ["product_id", "part_name", "mileage", "brand", "model", "scrape_date"]
    ].sort_values("mileage")
)

      product_id                               part_name    mileage   brand  \
11039   55170665                       Turbo aggregate -  2,000.000   skoda   
11280   54242387                      Suspension -(Rear)  2,000.000   skoda   
494     54247541          Drive shaft - , e-(Left front)  2,000.000      vw   
587     54254408  Trailing link rear - , e-(Right upper)  2,000.000      vw   
2474    54254408  Trailing link rear - , e-(Right upper)  2,000.000      vw   
...          ...                                     ...        ...     ...   
4526    59862593                       Kamera Utvändig - 10,000.000  toyota   
4488    59241763             Actuator loom -(Right rear) 10,000.000  toyota   
4266    59230184                       Suspension-(Rear) 10,000.000  toyota   
4265    59230183                      Suspension -(Rear) 10,000.000  toyota   
3962    59245086                    Airbag krocksensor - 10,000.000  toyota   

         model scrape_date  
11039  octavia  2026-0

## Additional Formatting Cleanup

These final formatting adjustments standardize a small number of categorical text fields without changing the overall structure of the dataset.

In [23]:
# Standardize quality grade formatting
df["quality_grade"] = df["quality_grade"].astype("string").str.strip().str.upper()

# Check cleaned value counts
print(df["quality_grade"].value_counts(dropna=False))

quality_grade
A2    9669
A1    1295
A3     178
B2     122
B1      18
C       13
C2       7
B3       7
C1       6
A        6
Name: count, dtype: Int64


In [24]:
# Standardize subcategory text
df["subcategory"] = df["subcategory"].astype("string").str.strip().str.lower()

# Check cleaned value counts
print(df["subcategory"].value_counts(dropna=False).head(20))

subcategory
all                               1048
rear                               315
right                              297
left                               277
right rear                         255
left rear                          249
left front                         200
right front                        199
either side                        128
passenger airbag                   101
rear axle beam                     100
steering wheel airbag              100
knee                               100
contact roll airbag                100
automatic gear                     100
abs hydraulic aggregate            100
brake servo                        100
fuse box / electricity central     100
abs hydraulic pump                 100
actuator loom                      100
Name: count, dtype: Int64


## Note on Subcategory

The `subcategory` field contains both true subtype labels and positional or location descriptors such as `left`, `rear`, and `right front`. It should therefore be interpreted as a mixed taxonomy field rather than a fully standardized part hierarchy.

## Validate Traficom Merge Outputs

This is a short post-merge sanity check only. The rebuilt first-registration features keep the same column names, so no extra cleaning logic is needed here unless merge keys or lifecycle columns look invalid.


In [25]:
lifecycle_columns = [
    "model_firstreg_total_2014_2026",
    "model_firstreg_recent_share",
    "model_firstreg_old_share",
    "model_firstreg_weighted_year",
    "model_firstreg_peak_year",
    "model_firstreg_peak_count",
    "model_firstreg_year_span",
    "brand_firstreg_total_2014_2026",
    "brand_firstreg_recent_share",
    "brand_firstreg_old_share",
    "brand_firstreg_weighted_year",
    "brand_firstreg_peak_year",
    "brand_firstreg_peak_count",
    "brand_firstreg_year_span",
]

present_lifecycle_columns = [column for column in lifecycle_columns if column in df.columns]
missing_lifecycle_columns = [column for column in lifecycle_columns if column not in df.columns]
known_model_families = {"golf", "corolla", "octavia"}
repair_status_values = set(df["repair_status"].dropna().astype("string")) if "repair_status" in df.columns else set()
known_brands = {"toyota", "skoda", "volkswagen"}

merge_sanity_df = pd.DataFrame(
    {
        "check": [
            "all lifecycle columns present",
            "brand_merge_key looks like model family",
            "model_merge_key looks like repair status",
            "model_merge_key looks like brand",
        ],
        "value": [
            len(missing_lifecycle_columns) == 0,
            int(df["brand_merge_key"].astype("string").isin(known_model_families).sum()) if "brand_merge_key" in df.columns else pd.NA,
            int(df["model_merge_key"].astype("string").isin(repair_status_values).sum()) if "model_merge_key" in df.columns else pd.NA,
            int(df["model_merge_key"].astype("string").isin(known_brands).sum()) if "model_merge_key" in df.columns else pd.NA,
        ],
    }
)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
print("Present lifecycle columns:", present_lifecycle_columns)
print("Missing lifecycle columns:", missing_lifecycle_columns)
display(merge_sanity_df)


Rows: 11,321
Columns: 71
Present lifecycle columns: ['model_firstreg_total_2014_2026', 'model_firstreg_recent_share', 'model_firstreg_old_share', 'model_firstreg_weighted_year', 'model_firstreg_peak_year', 'model_firstreg_peak_count', 'model_firstreg_year_span', 'brand_firstreg_total_2014_2026', 'brand_firstreg_recent_share', 'brand_firstreg_old_share', 'brand_firstreg_weighted_year', 'brand_firstreg_peak_year', 'brand_firstreg_peak_count', 'brand_firstreg_year_span']
Missing lifecycle columns: []


,check,value
0,all lifecycle columns present,True
1,brand_merge_key looks like model family,0
2,model_merge_key looks like repair status,0
3,model_merge_key looks like brand,0


In [26]:
if present_lifecycle_columns:
    lifecycle_missing_summary = pd.DataFrame({
        "missing_count": df[present_lifecycle_columns].isna().sum(),
        "missing_percentage": df[present_lifecycle_columns].isna().mean() * 100,
    })
    display(lifecycle_missing_summary)
else:
    print("No lifecycle columns available for missing-value summary.")


,missing_count,missing_percentage
model_firstreg_total_2014_2026,0,0.000
model_firstreg_recent_share,0,0.000
model_firstreg_old_share,0,0.000
model_firstreg_weighted_year,0,0.000
model_firstreg_peak_year,0,0.000
model_firstreg_peak_count,0,0.000
model_firstreg_year_span,0,0.000
brand_firstreg_total_2014_2026,0,0.000
brand_firstreg_recent_share,0,0.000
brand_firstreg_old_share,0,0.000


In [27]:
# Preview lifecycle values for target model families.
target_model_values = ["golf", "corolla", "octavia"]
target_key_column = "model_family_clean" if "model_family_clean" in df.columns else "model"
target_preview_columns = [
    column
    for column in ["brand", "model", "model_family_clean"] + lifecycle_columns
    if column in df.columns
]

target_lifecycle_preview = (
    df.loc[df[target_key_column].isin(target_model_values), target_preview_columns]
    .drop_duplicates()
    .sort_values([column for column in ["brand", target_key_column] if column in target_preview_columns])
)

display(target_lifecycle_preview.head(20))


,brand,model,model_family_clean,model_firstreg_total_2014_2026,model_firstreg_recent_share,model_firstreg_old_share,model_firstreg_weighted_year,model_firstreg_peak_year,model_firstreg_peak_count,model_firstreg_year_span,brand_firstreg_total_2014_2026,brand_firstreg_recent_share,brand_firstreg_old_share,brand_firstreg_weighted_year,brand_firstreg_peak_year,brand_firstreg_peak_count,brand_firstreg_year_span
7684,skoda,octavia,octavia,"48,713.000",0.232,0.573,"2,018.140","2,014.000","5,868.000",11.000,"93,798.000",0.411,0.363,"2,020.009","2,018.000","11,697.000",9.000
3790,toyota,corolla,corolla,"34,912.000",0.599,0.103,"2,020.982","2,020.000","5,395.000",11.000,"137,879.000",0.485,0.308,"2,020.379","2,019.000","15,184.000",9.000
0,vw,golf,golf,"31,752.000",0.167,0.671,"2,017.576","2,014.000","5,265.000",11.000,"100,381.000",0.393,0.391,"2,019.842","2,018.000","13,274.000",9.000


In [28]:
# Confirm this validation section has not changed the number of price rows.
print(f"Raw merged rows: {df_raw.shape[0]:,}")
print(f"Current rows: {df.shape[0]:,}")
print("Lifecycle validation is diagnostic only; it should not change the number of price rows.")


Raw merged rows: 11,374
Current rows: 11,321
Lifecycle validation is diagnostic only; it should not change the number of price rows.


## Add Listing-History Features

These exploratory listing-history variables summarize how often each `product_id` appears in the sampled marketplace window and whether its observed listing price changed across scrape dates. They are computed from repeated marketplace observations in the cleaned dataframe and merged back before saving the final clean master dataset.

In [29]:
# Check required columns and parse scrape_date for listing-history features.
listing_history_required_columns = ["product_id", "scrape_date", "price"]
missing_listing_history_columns = [
    column for column in listing_history_required_columns if column not in df.columns
]

if missing_listing_history_columns:
    print(f"Cannot create listing-history features. Missing columns: {missing_listing_history_columns}")
    can_build_listing_history = False
else:
    can_build_listing_history = True
    df["scrape_date"] = pd.to_datetime(df["scrape_date"], errors="coerce")
    print("Required listing-history columns are present.")
    print(f"Unparseable scrape_date values: {df['scrape_date'].isna().sum():,}")

Required listing-history columns are present.
Unparseable scrape_date values: 0


In [30]:
# Engineer listing-history variables: legacy full-history columns for exploratory analysis
# plus new point-in-time-safe row-level columns for trusted model training.
listing_history_columns = [
    "times_observed",
    "first_seen_date",
    "last_seen_date",
    "observed_span_days",
    "price_changed_flag",
    "price_change_count",
    "absolute_price_change",
    "relative_price_change_pct",
]

listing_history_safe_columns = [
    "observations_so_far",
    "days_since_first_seen_so_far",
    "price_changed_flag_so_far",
    "price_change_count_so_far",
    "absolute_price_change_so_far",
    "relative_price_change_pct_so_far",
]

def count_price_changes(values):
    prices = values.dropna()
    if len(prices) <= 1:
        return 0
    return int(prices.ne(prices.shift()).iloc[1:].sum())

def first_non_null(values):
    values = values.dropna()
    return values.iloc[0] if len(values) else np.nan

def last_non_null(values):
    values = values.dropna()
    return values.iloc[-1] if len(values) else np.nan

if can_build_listing_history:
    listing_history_shape_before = df.shape
    listing_history_work = df[["product_id", "scrape_date", "price"]].copy()
    listing_history_work["price"] = pd.to_numeric(listing_history_work["price"], errors="coerce")
    listing_history_work["_row_order"] = np.arange(len(listing_history_work))
    listing_history_work = listing_history_work.sort_values(
        ["product_id", "scrape_date", "_row_order"],
        kind="mergesort",
        na_position="last",
    )

    # Legacy full-history aggregates remain useful for exploratory comparison only.
    listing_history_features = listing_history_work.groupby("product_id", as_index=False).agg(
        times_observed=("product_id", "size"),
        first_seen_date=("scrape_date", "min"),
        last_seen_date=("scrape_date", "max"),
        price_changed_flag=("price", lambda values: int(values.nunique(dropna=True) > 1)),
        price_change_count=("price", count_price_changes),
        first_observed_price=("price", first_non_null),
        last_observed_price=("price", last_non_null),
    )

    listing_history_features["observed_span_days"] = (
        listing_history_features["last_seen_date"] - listing_history_features["first_seen_date"]
    ).dt.days
    listing_history_features["absolute_price_change"] = (
        listing_history_features["last_observed_price"] - listing_history_features["first_observed_price"]
    )
    listing_history_features["relative_price_change_pct"] = np.where(
        listing_history_features["first_observed_price"].notna()
        & listing_history_features["first_observed_price"].ne(0),
        listing_history_features["absolute_price_change"] / listing_history_features["first_observed_price"] * 100,
        np.nan,
    )
    listing_history_features = listing_history_features[["product_id"] + listing_history_columns]

    # Point-in-time-safe row-level history uses only information available up to each scrape_date.
    listing_history_point_in_time = listing_history_work[["product_id", "scrape_date", "price", "_row_order"]].copy()
    grouped_listing_history = listing_history_point_in_time.groupby("product_id", sort=False)
    first_seen_so_far = grouped_listing_history["scrape_date"].transform("first")
    first_observed_price_so_far = grouped_listing_history["price"].transform("first")
    previous_price = grouped_listing_history["price"].shift(1)

    listing_history_point_in_time["observations_so_far"] = grouped_listing_history.cumcount() + 1
    listing_history_point_in_time["days_since_first_seen_so_far"] = (
        listing_history_point_in_time["scrape_date"] - first_seen_so_far
    ).dt.days
    listing_history_point_in_time["price_change_event_so_far"] = (
        previous_price.notna()
        & listing_history_point_in_time["price"].notna()
        & listing_history_point_in_time["price"].ne(previous_price)
    ).astype(int)
    listing_history_point_in_time["price_change_count_so_far"] = (
        listing_history_point_in_time.groupby("product_id", sort=False)["price_change_event_so_far"].cumsum()
    )
    listing_history_point_in_time["price_changed_flag_so_far"] = (
        listing_history_point_in_time["price_change_count_so_far"] > 0
    ).astype(int)
    listing_history_point_in_time["absolute_price_change_so_far"] = (
        listing_history_point_in_time["price"] - first_observed_price_so_far
    )
    listing_history_point_in_time["relative_price_change_pct_so_far"] = np.where(
        first_observed_price_so_far.notna() & first_observed_price_so_far.ne(0),
        listing_history_point_in_time["absolute_price_change_so_far"] / first_observed_price_so_far * 100,
        np.nan,
    )

    listing_history_point_in_time_features = (
        listing_history_point_in_time[["_row_order"] + listing_history_safe_columns]
        .sort_values("_row_order")
        .reset_index(drop=True)
    )

    print(f"Dataframe shape before listing-history merge: {listing_history_shape_before}")
    print("Legacy exploratory listing-history columns:", listing_history_columns)
    print("New point-in-time-safe listing-history columns:", listing_history_safe_columns)
    display(listing_history_features.head())
    display(listing_history_point_in_time_features.head())
else:
    listing_history_features = pd.DataFrame(columns=["product_id"] + listing_history_columns)
    listing_history_point_in_time_features = pd.DataFrame(columns=["_row_order"] + listing_history_safe_columns)

Dataframe shape before listing-history merge: (11321, 71)
Legacy exploratory listing-history columns: ['times_observed', 'first_seen_date', 'last_seen_date', 'observed_span_days', 'price_changed_flag', 'price_change_count', 'absolute_price_change', 'relative_price_change_pct']
New point-in-time-safe listing-history columns: ['observations_so_far', 'days_since_first_seen_so_far', 'price_changed_flag_so_far', 'price_change_count_so_far', 'absolute_price_change_so_far', 'relative_price_change_pct_so_far']


,product_id,times_observed,first_seen_date,last_seen_date,observed_span_days,price_changed_flag,price_change_count,absolute_price_change,relative_price_change_pct
0,53365006,1,2026-02-03,2026-02-03,0,0,0,0.000,0.000
1,53365016,1,2026-02-03,2026-02-03,0,0,0,0.000,0.000
2,53365106,6,2026-02-03,2026-02-18,15,1,5,-1.000,-0.111
3,53366348,6,2026-02-03,2026-02-18,15,1,4,-0.100,-0.211
4,53367213,6,2026-02-03,2026-02-18,15,1,5,-0.400,-0.109


,_row_order,observations_so_far,days_since_first_seen_so_far,price_changed_flag_so_far,price_change_count_so_far,absolute_price_change_so_far,relative_price_change_pct_so_far
0,0,1,0,0,0,0.000,0.000
1,1,1,0,0,0,0.000,0.000
2,2,1,0,0,0,0.000,0.000
3,3,1,0,0,0,0.000,0.000
4,4,1,0,0,0,0.000,0.000


In [31]:
# Merge listing-history variables back and validate the result.
if can_build_listing_history:
    existing_listing_history_columns = [column for column in listing_history_columns if column in df.columns]
    existing_listing_history_safe_columns = [
        column for column in listing_history_safe_columns if column in df.columns
    ]
    columns_to_drop = list(dict.fromkeys(
        existing_listing_history_columns + existing_listing_history_safe_columns
    ))
    if columns_to_drop:
        df = df.drop(columns=columns_to_drop)

    rows_before_listing_history_merge = len(df)

    # Row-level safe history aligns one-to-one with the original dataframe order.
    df[listing_history_safe_columns] = listing_history_point_in_time_features[
        listing_history_safe_columns
    ].to_numpy()

    # Legacy full-history columns are merged only for exploratory diagnostics.
    df = df.merge(listing_history_features, on="product_id", how="left", validate="many_to_one")

    print(f"Dataframe shape before listing-history merge: {listing_history_shape_before}")
    print(f"Dataframe shape after listing-history merge: {df.shape}")
    print("Newly added legacy exploratory columns:", listing_history_columns)
    print("Newly added point-in-time-safe columns:", listing_history_safe_columns)

    if len(df) == rows_before_listing_history_merge:
        print("Row count unchanged after listing-history merge.")
    else:
        print("Warning: row count changed after listing-history merge.")

    print("Null counts for legacy and safe listing-history columns:")
    display(df[listing_history_columns + listing_history_safe_columns].isna().sum().to_frame("null_count"))

    print("Summary for point-in-time-safe listing-history columns:")
    display(df[list(listing_history_safe_columns)].describe())

    print("Summary for legacy exploratory listing-history count/span columns:")
    display(df[["times_observed", "observed_span_days", "price_change_count"]].describe())

    print("Value counts for price-changed flags:")
    display(
        pd.concat(
            {
                "price_changed_flag": df["price_changed_flag"].value_counts(dropna=False),
                "price_changed_flag_so_far": df["price_changed_flag_so_far"].value_counts(dropna=False),
            },
            axis=1,
        )
    )

    repeated_product_ids = df.loc[df["times_observed"] > 1, "product_id"].drop_duplicates().head(5)
    sample_columns = [
        "product_id",
        "scrape_date",
        "price",
        "observations_so_far",
        "days_since_first_seen_so_far",
        "price_change_count_so_far",
        "absolute_price_change_so_far",
        "relative_price_change_pct_so_far",
        "times_observed",
        "observed_span_days",
        "price_change_count",
        "absolute_price_change",
        "relative_price_change_pct",
    ]
    print("Sample repeated product_id cases with safe vs exploratory history:")
    display(
        df.loc[df["product_id"].isin(repeated_product_ids), sample_columns]
        .sort_values(["product_id", "scrape_date"], kind="mergesort")
        .head(30)
    )

    print("Extreme point-in-time-safe relative price change values:")
    display(df["relative_price_change_pct_so_far"].describe())
    display(
        df[["product_id", "scrape_date", "relative_price_change_pct_so_far"]]
        .sort_values("relative_price_change_pct_so_far")
        .head(5)
    )
    display(
        df[["product_id", "scrape_date", "relative_price_change_pct_so_far"]]
        .sort_values("relative_price_change_pct_so_far", ascending=False)
        .head(5)
    )

Dataframe shape before listing-history merge: (11321, 71)
Dataframe shape after listing-history merge: (11321, 85)
Newly added legacy exploratory columns: ['times_observed', 'first_seen_date', 'last_seen_date', 'observed_span_days', 'price_changed_flag', 'price_change_count', 'absolute_price_change', 'relative_price_change_pct']
Newly added point-in-time-safe columns: ['observations_so_far', 'days_since_first_seen_so_far', 'price_changed_flag_so_far', 'price_change_count_so_far', 'absolute_price_change_so_far', 'relative_price_change_pct_so_far']
Row count unchanged after listing-history merge.
Null counts for legacy and safe listing-history columns:


,null_count
times_observed,0
first_seen_date,0
last_seen_date,0
observed_span_days,0
price_changed_flag,0
price_change_count,0
absolute_price_change,0
relative_price_change_pct,0
observations_so_far,0
days_since_first_seen_so_far,0


Summary for point-in-time-safe listing-history columns:


,observations_so_far,days_since_first_seen_so_far,price_changed_flag_so_far,price_change_count_so_far,absolute_price_change_so_far,relative_price_change_pct_so_far
count,"11,321.000","11,321.000","11,321.000","11,321.000","11,321.000","11,321.000"
mean,3.281,6.883,0.762,1.935,-0.036,-0.013
std,1.759,5.300,0.426,1.581,2.043,0.412
min,1.000,0.000,0.000,0.000,-35.400,-29.899
25%,2.000,3.000,1.000,1.000,-0.200,-0.322
50%,3.000,6.000,1.000,2.000,0.000,0.000
75%,5.000,12.000,1.000,3.000,0.200,0.169
max,6.000,15.000,1.000,5.000,23.400,0.933


Summary for legacy exploratory listing-history count/span columns:


...

,product_id,scrape_date,price,observations_so_far,days_since_first_seen_so_far,price_change_count_so_far,absolute_price_change_so_far,relative_price_change_pct_so_far,times_observed,observed_span_days,price_change_count,absolute_price_change,relative_price_change_pct
5,54655558,2026-02-03,23.700,1.000,0.000,0.000,0.000,0.000,6,15,3,0.000,0.000
639,54655558,2026-02-06,23.600,2.000,3.000,1.000,-0.100,-0.422,6,15,3,0.000,0.000
1272,54655558,2026-02-09,23.600,3.000,6.000,1.000,-0.100,-0.422,6,15,3,0.000,0.000
1892,54655558,2026-02-12,23.800,4.000,9.000,2.000,0.100,0.422,6,15,3,0.000,0.000
2526,54655558,2026-02-15,23.700,5.000,12.000,3.000,0.000,0.000,6,15,3,0.000,0.000
3160,54655558,2026-02-18,23.700,6.000,15.000,3.000,0.000,0.000,6,15,3,0.000,0.000
2,58952159,2026-02-03,177.600,1.000,0.000,0.000,0.000,0.000,6,15,5,-0.200,-0.113
635,58952159,2026-02-06,177.000,2.000,3.000,1.000,-0.600,-0.338,6,15,5,-0.200,-0.113
1268,58952159,2026-02-09,176.900,3.000,6.000,2.000,-0.700,-0.394,6,15,5,-0.200,-0.113
1889,58952159,2026-02-12,178.500,4.000,9.000,3.000,0.900,0.507,6,15,5,-0.200,-0.113


Extreme point-in-time-safe relative price change values:


count   11,321.000
mean        -0.013
std          0.412
min        -29.899
25%         -0.322
50%          0.000
75%          0.169
max          0.933
Name: relative_price_change_pct_so_far, dtype: float64

,product_id,scrape_date,relative_price_change_pct_so_far
10848,62801577,2026-02-18,-29.899
11106,66697123,2026-02-18,-0.663
3206,54223147,2026-02-18,-0.636
3164,54440014,2026-02-18,-0.630
1526,54427789,2026-02-09,-0.625


,product_id,scrape_date,relative_price_change_pct_so_far
6620,66735158,2026-02-12,0.933
9612,64797981,2026-02-12,0.895
10041,60226014,2026-02-12,0.879
1901,63199755,2026-02-12,0.866
10048,66730848,2026-02-12,0.865


## Save Cleaned Dataset

The file saved here is the final cleaned master dataset for later splitting and modeling. It includes conservative formatting/type fixes, Traficom lifecycle variables, and the listing-history features added above.

In [32]:
output_path = Path("/Users/riteshbhandari/Documents/Dokumentit – Ritesh - MacBook Pro/GitHub/DPPM/datasets/cleaned/clean_master_dataset.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(output_path, index=False)
print(f"Clean master dataset saved to: {output_path}")


Clean master dataset saved to: /Users/riteshbhandari/Documents/Dokumentit – Ritesh - MacBook Pro/GitHub/DPPM/datasets/cleaned/clean_master_dataset.csv
